# Notebook 74: Individual cases of interest

In [11]:
nb_id = 74

In [12]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import mesa_reader as mr

plt.style.use('./plotstyle.mplstyle')

import sys
sys.path.append('..')
from src.util import DATA_DIR, MESA_DATA_DIR
from src.constants import Z_SUN

In [13]:
FIGURE_FOLDER = Path(f'./output/nb{nb_id}/figures')
FIGURE_FOLDER.mkdir(parents=True, exist_ok=True)

In [58]:
project_folder = MESA_DATA_DIR / '00_fiducial' / '0005_ZdivZsun_1d-01'
list(project_folder.glob('00*'))

[PosixPath('/mnt/ceph/users/ldesa/mesa_che_grids/sse_production/00_fiducial/0005_ZdivZsun_1d-01/006_m35'),
 PosixPath('/mnt/ceph/users/ldesa/mesa_che_grids/sse_production/00_fiducial/0005_ZdivZsun_1d-01/003_m20'),
 PosixPath('/mnt/ceph/users/ldesa/mesa_che_grids/sse_production/00_fiducial/0005_ZdivZsun_1d-01/007_m40'),
 PosixPath('/mnt/ceph/users/ldesa/mesa_che_grids/sse_production/00_fiducial/0005_ZdivZsun_1d-01/009_m50'),
 PosixPath('/mnt/ceph/users/ldesa/mesa_che_grids/sse_production/00_fiducial/0005_ZdivZsun_1d-01/008_m45'),
 PosixPath('/mnt/ceph/users/ldesa/mesa_che_grids/sse_production/00_fiducial/0005_ZdivZsun_1d-01/001_m10'),
 PosixPath('/mnt/ceph/users/ldesa/mesa_che_grids/sse_production/00_fiducial/0005_ZdivZsun_1d-01/004_m25'),
 PosixPath('/mnt/ceph/users/ldesa/mesa_che_grids/sse_production/00_fiducial/0005_ZdivZsun_1d-01/002_m15'),
 PosixPath('/mnt/ceph/users/ldesa/mesa_che_grids/sse_production/00_fiducial/0005_ZdivZsun_1d-01/005_m30')]

In [59]:
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

target_teff = 177e3
target_logL = 6.05
target_m = 20.
target_he = 0.90

# Build the list of directories first
dirs = []
for d1 in project_folder.glob('[0-9][0-9][0-9]_m*'):
    for d2 in d1.glob('m*_p*_w*/'):
        if d2.is_dir():
            dirs.append(d2)

def process_dir(d2):
    """Process one model directory and return dict of arrays or None."""
    h = mr.MesaData(str(d2 / 'LOGS' / 'history.data'))

    try:
        match_i = np.where(
            #(np.isclose(10.**h.log_Teff, target_teff, atol=50)) &
            (np.isclose(h.log_L, target_logL, atol=0.2)) &
            (np.isclose(h.star_mass, target_m, atol=10.0)) &
            (np.isclose(h.surface_he4, target_he, atol=0.1))
        )[0]
        if len(match_i) > 1:
            match_i = [match_i[len(match_i)//2]]
    except IndexError:
        return None

    if len(match_i) == 0:
        return None

    # Wrap all outputs in np.atleast_1d to guarantee 1D arrays for concatenation
    return {
        'm': np.atleast_1d(h.star_mass[match_i]),
        'logL': np.atleast_1d(h.log_L[match_i]),
        'R': np.atleast_1d(h.radius[match_i]),
        'Teff': np.atleast_1d(10**h.log_Teff[match_i]),
        'Mdot': np.atleast_1d(h.log_abs_mdot[match_i]),
        'age': np.atleast_1d(h.star_age[match_i]),
        'Mzams': np.atleast_1d(h.star_mass[0]),
        'vrot_zams': np.atleast_1d(h.surf_avg_v_rot[0]),
        'center_h': np.atleast_1d(h.center_h1[match_i]),
        'x_he': np.atleast_1d(h.surface_he4[match_i]),
        'x_c': np.atleast_1d(h.surface_c12[match_i]),
        'x_n': np.atleast_1d(h.surface_n14[match_i]),
        'x_o': np.atleast_1d(h.surface_o16[match_i]),
        'x_ne': np.atleast_1d(h.surface_ne20[match_i]),
        'x_fe': np.atleast_1d(h.surface_fe56[match_i]),
        'path': str(d2),
    }

# Parallel execution
results = []
with ProcessPoolExecutor(max_workers=96) as executor:
    for res in executor.map(process_dir, dirs):
        if res is not None:
            results.append(res)

# Concatenate all results into arrays
def collect(key):
    return np.concatenate([r[key] for r in results])

match_m         = collect('m')
match_logL      = collect('logL')
match_R         = collect('R')
match_Teff      = collect('Teff')
match_Mdot      = collect('Mdot')
match_age       = collect('age')
match_Mzams     = collect('Mzams')
match_vrot_zams = collect('vrot_zams')
match_center_h  = collect('center_h')
match_x_he      = collect('x_he')
match_x_c       = collect('x_c')
match_x_n       = collect('x_n')
match_x_o       = collect('x_o')
match_x_ne      = collect('x_ne')
match_x_fe      = collect('x_fe')

print(f"Got {len(match_m)} matches in total")



Got 5 matches in total


In [60]:
df_columns = [
    'Mzams', 'vrot_zams', 'm', 'logL', 'R', 'Teff', 'Mdot', 'age',
    'center_h', 'x_he', 'x_c', 'x_n', 'x_o', 'x_ne', 'x_fe', 'path'
]
import pandas as pd
df = pd.DataFrame({col: collect(col) for col in df_columns if col != 'path'
})

In [61]:
pd.set_option('display.max_rows', 500)

In [62]:
df

,Mzams,vrot_zams,m,logL,R,Teff,Mdot,age,center_h,x_he,x_c,x_n,x_o,x_ne,x_fe
0,35.0,0.0,29.389605,5.996776,1.769024,136978.903160,-5.046617,7.016347e+06,1.512321e-18,0.938268,0.018922,0.030841,0.002771,0.000129,3.646716e-81
1,35.0,0.0,29.331638,5.989869,1.781847,135943.521510,-5.055797,7.106804e+06,1.862649e-18,0.946860,0.015510,0.027205,0.002105,0.000128,4.343705e-81
2,25.0,0.0,22.962263,5.884976,3.072012,97467.359264,-4.433310,9.767823e+06,2.229988e-19,0.943412,0.000023,0.001068,0.000010,0.000125,1.541924e-99
3,30.0,0.0,26.878049,5.921464,2.008239,123107.609213,-5.474722,8.035970e+06,3.292359e-18,0.958461,0.000016,0.001170,0.000011,0.000125,2.678643e-96
4,30.0,0.0,26.697991,5.911054,1.969640,123565.316598,-5.426257,8.145796e+06,3.780453e-18,0.968769,0.000018,0.001252,0.000012,0.000125,9.791868e-95
